# Lab 01: API Testing — Sandboxes & Test Cards

**Duration**: ~10 minutes  
**Prerequisites**: Completed `01_api_testing_intro.ipynb`

## Learning Objectives

By the end of this notebook, you will:
- Create payments using test cards
- Simulate successful and declined transactions
- Implement proper error handling for payment failures

---

In [6]:
!pip install stripe --quiet

import os
import stripe

try:
    from google.colab import userdata
    stripe.api_key = userdata.get('STRIPE_SECRET_KEY')
    print("Loaded API key from Colab Secrets.")
except Exception:
    import getpass
    stripe.api_key = getpass.getpass("Paste your Stripe test secret key (sk_test_...): ")

try:
    account = stripe.Account.retrieve()
    mode = 'Sandbox' if 'test' in stripe.api_key else 'LIVE — BE CAREFUL!'
    print(f"Connected to account: {account.id} | Mode: {mode}")
except stripe.error.AuthenticationError:
    print("ERROR: Invalid API key.")

Loaded API key from Colab Secrets.
Connected to account: acct_1RnL4mBMxfUzotEq | Mode: Sandbox


---

## Test Cards Reference

Stripe provides special card numbers for testing. These only work with `sk_test_` keys.

### Successful Payment Cards

| Brand | Number | PaymentMethod identifier |
|-------|--------|---------------------------|
| Visa | 4242 4242 4242 4242 | `pm_card_visa` |
| Visa (debit) | 4000 0566 5566 5556 | `pm_card_visa_debit` |
| Mastercard | 5555 5555 5555 4444 | `pm_card_mastercard` |
| American Express | 3782 822463 10005 | `pm_card_amex` |

Any future expiry date, any CVC, any ZIP work with all test cards.

> **In code, use `PaymentMethod` identifiers (`pm_card_visa`) rather than raw card numbers.** This keeps your code PCI compliant and mirrors the production pattern where card data is collected by Stripe Elements or Checkout.

---

## Exercise 1: Create a Successful Payment

In [7]:
payment_intent = stripe.PaymentIntent.create(
    amount=2000,  # $20.00 in cents
    currency="usd",
    payment_method="pm_card_visa",
    confirm=True,
    automatic_payment_methods={"enabled": True, "allow_redirects": "never"}
)

print(f"Payment ID: {payment_intent.id}")
print(f"Amount:     ${payment_intent.amount / 100:.2f} {payment_intent.currency.upper()}")
print(f"Status:     {payment_intent.status}")

# Expected: Status = succeeded

Payment ID: pi_3TMUaHBMxfUzotEq27rvt1YL
Amount:     $20.00 USD
Status:     succeeded


**Dashboard**: Open [Payments](https://dashboard.stripe.com/test/payments) to see your test payment.

---

## Exercise 2: Simulate Declined Payments

### Common Decline Scenarios

| Scenario | PaymentMethod identifier |
|----------|---------------------------|
| Generic decline | `pm_card_visa_chargeDeclined` |
| Insufficient funds | `pm_card_visa_chargeDeclinedInsufficientFunds` |
| Lost card | `pm_card_visa_chargeDeclinedLostCard` |
| Stolen card | `pm_card_visa_chargeDeclinedStolenCard` |
| Expired card | `pm_card_chargeDeclinedExpiredCard` |
| Incorrect CVC | `pm_card_chargeDeclinedIncorrectCvc` |

In [8]:
# Without error handling this raises an exception — let's see what it looks like
try:
    stripe.PaymentIntent.create(
        amount=1500,
        currency="usd",
        payment_method="pm_card_visa_chargeDeclined",
        confirm=True,
        automatic_payment_methods={"enabled": True, "allow_redirects": "never"}
    )
except stripe.error.CardError as e:
    print(f"Payment declined!")
    print(f"  Error code:   {e.code}")
    print(f"  Decline code: {e.error.decline_code}")
    print(f"  Message:      {e.user_message}")

Payment declined!
  Error code:   card_declined
  Decline code: generic_decline
  Message:      Your card was declined.


---

## Exercise 3: Production-Ready Error Handling

Your integration must handle all failure modes. Here is the complete error hierarchy:

| Exception | Cause | User action |
|-----------|-------|-------------|
| `CardError` | Card declined / invalid | Ask for a different card |
| `InvalidRequestError` | Bad API parameters | Fix your code |
| `AuthenticationError` | Invalid API key | Check configuration |
| `RateLimitError` | Too many requests | Retry with backoff |
| `StripeError` | General Stripe error | Retry or contact support |

In [9]:
def create_payment(amount_cents: int, currency: str, payment_method: str) -> dict:
    """
    Create a payment with comprehensive error handling.

    Returns a dict with keys:
      success (bool)
      payment_intent (on success)
      error (on failure): dict with type, code, decline_code, message
    """
    try:
        pi = stripe.PaymentIntent.create(
            amount=amount_cents,
            currency=currency,
            payment_method=payment_method,
            confirm=True,
            automatic_payment_methods={"enabled": True, "allow_redirects": "never"}
        )
        return {"success": True, "payment_intent": pi}

    except stripe.error.CardError as e:
        return {"success": False, "error": {
            "type": "card_error",
            "code": e.code,
            "decline_code": getattr(e.error, "decline_code", None),
            "message": e.user_message
        }}

    except stripe.error.InvalidRequestError as e:
        return {"success": False, "error": {"type": "invalid_request", "message": str(e)}}

    except stripe.error.AuthenticationError:
        return {"success": False, "error": {"type": "authentication_error", "message": "Invalid API key"}}

    except stripe.error.StripeError as e:
        return {"success": False, "error": {"type": "stripe_error", "message": str(e)}}

print("create_payment() defined.")

create_payment() defined.


In [10]:
# Run through multiple scenarios
test_cases = [
    ("Successful Visa",       "pm_card_visa",                                   1000),
    ("Declined (generic)",    "pm_card_visa_chargeDeclined",                    1500),
    ("Insufficient funds",    "pm_card_visa_chargeDeclinedInsufficientFunds",   2000),
    ("Successful Mastercard", "pm_card_mastercard",                             2500),
]

print("Testing Payment Scenarios")
print("=" * 60)

for name, pm, amount in test_cases:
    result = create_payment(amount, "usd", pm)
    print(f"\n{name}:")
    if result["success"]:
        pi = result["payment_intent"]
        print(f"  Status: SUCCESS | ID: {pi.id} | Amount: ${pi.amount / 100:.2f}")
    else:
        err = result["error"]
        print(f"  Status: FAILED  | Type: {err['type']} | {err['message']}")
        if err.get("decline_code"):
            print(f"  Decline code: {err['decline_code']}")

Testing Payment Scenarios

Successful Visa:
  Status: SUCCESS | ID: pi_3TMUaKBMxfUzotEq1NMU1qUk | Amount: $10.00

Declined (generic):
  Status: FAILED  | Type: card_error | Your card was declined.
  Decline code: generic_decline

Insufficient funds:
  Status: FAILED  | Type: card_error | Your card has insufficient funds.
  Decline code: insufficient_funds

Successful Mastercard:
  Status: SUCCESS | ID: pi_3TMUaNBMxfUzotEq1PCJCR2j | Amount: $25.00


### Checkpoint

You should see:
- 2 successful payments (Visa and Mastercard)
- 2 declined payments with specific decline codes

**Dashboard**: [Payments](https://dashboard.stripe.com/test/payments) shows both successful and failed attempts.

---

## Best Practices for Test Cards

1. **Use `pm_card_*` identifiers in code**, never raw card numbers
2. **Test every failure scenario** your users might hit: declines, expired cards, wrong CVC
3. **Map decline codes to user-friendly messages:**

```python
DECLINE_MESSAGES = {
    "generic_decline":      "Your card was declined. Please try another card.",
    "insufficient_funds":   "Insufficient funds. Please try another card.",
    "expired_card":         "Your card has expired. Please update your details.",
    "incorrect_cvc":        "Invalid security code. Please check and try again.",
}
```

4. **Log the full error object** server-side for debugging while showing friendly messages to users.

---

## Summary

- **Test cards** simulate real payment scenarios without moving money
- **`pm_card_*` identifiers** are the PCI-compliant way to reference cards in test code
- Always catch `stripe.error.CardError` and the full exception hierarchy
- **Decline codes** let you give specific, actionable feedback to users

## Next Steps

Open `03_test_clocks.ipynb` to fast-forward through subscription lifecycles.